In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import pandas as pd

# 1. Read the Excel file into a Pandas DataFrame
pdf = pd.read_excel("/Volumes/my_projects/default/teleco_churn_analysis/Telco_customer_churn.xlsx")

# 2. Clean the 'Total Charges' column - replace spaces with NaN and convert to numeric
pdf['Total Charges'] = pd.to_numeric(pdf['Total Charges'], errors='coerce')

# 3. Convert to a PySpark DataFrame
df = spark.createDataFrame(pdf)

# 3. View the schema or contents
df.printSchema()
display(df.limit(5))

root
 |-- CustomerID: string (nullable = true)
 |-- Count: long (nullable = true)
 |-- Country: string (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zip Code: long (nullable = true)
 |-- Lat Long: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Senior Citizen: string (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Tenure Months: long (nullable = true)
 |-- Phone Service: string (nullable = true)
 |-- Multiple Lines: string (nullable = true)
 |-- Internet Service: string (nullable = true)
 |-- Online Security: string (nullable = true)
 |-- Online Backup: string (nullable = true)
 |-- Device Protection: string (nullable = true)
 |-- Tech Support: string (nullable = true)
 |-- Streaming TV: string (nullable = true)
 |-- Streaming Movies: string (nullable = true)
 |-- Contract: string (

CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.30742,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes,1,67,2701,Moved
9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8,3046.05,Yes,1,84,5003,Moved
0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.7,5036.3,Yes,1,89,5340,Competitor had better devices


In [0]:
from pyspark.sql import functions as F
from scipy.stats import chi2_contingency
import numpy as np


def categorical_association(df, col1, col2):
    """
    Calculate association between two categorical columns.

    Returns:
        - PySpark cross-tab
        - Chi-square statistic
        - p-value
        - degrees of freedom
        - Cramer's V
    """

    # --------------------------------------------------
    # 1. Remove NULL values
    # --------------------------------------------------
    clean_df = df.select(col1, col2).dropna()

    # --------------------------------------------------
    # 2. Create cross-tab using PySpark
    # --------------------------------------------------
    crosstab_df = (
        clean_df
        .groupBy(col1)
        .pivot(col2)
        .count()
        .fillna(0)
    )

    # Display cross-tab
    crosstab_df.show()

    # --------------------------------------------------
    # 3. Convert ONLY the small contingency table
    #    to NumPy for scipy calculation
    # --------------------------------------------------

    # Get category columns
    category_columns = [
        c for c in crosstab_df.columns
        if c != col1
    ]

    # Collect only aggregated counts
    rows = crosstab_df.select(category_columns).collect()

    observed = np.array(
        [[row[c] for c in category_columns] for row in rows],
        dtype=float
    )

    # --------------------------------------------------
    # 4. Chi-square test
    # --------------------------------------------------
    chi2, p_value, dof, expected = chi2_contingency(observed)

    # --------------------------------------------------
    # 5. Cramer's V
    # --------------------------------------------------
    n = observed.sum()

    r, k = observed.shape

    cramers_v = np.sqrt(
        chi2 / (n * min(k - 1, r - 1))
    )

    # --------------------------------------------------
    # 6. Return results
    # --------------------------------------------------
    return {
        "column_1": col1,
        "column_2": col2,
        "chi_square": chi2,
        "p_value": p_value,
        "degrees_of_freedom": dof,
        "cramers_v": cramers_v,
        "sample_size": int(n)
    }

In [0]:
Qualitative_Cols = ["Gender", "Senior Citizen", "Partner", "Dependents", "Phone Service", "Multiple Lines", "Internet Service", "Online Security", "Online Backup", "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies", "Contract", "Paperless Billing", "Payment Method"]
Target_Cols = ["Churn Label"]
Feature_Cols = []

In [0]:
for i in range(len(Qualitative_Cols)):
    result = categorical_association(
        df,
        Qualitative_Cols[i],
        "Churn Label"
    )
    Feature_Cols.append(result)


+------+----+---+
|Gender|  No|Yes|
+------+----+---+
|Female|2549|939|
|  Male|2625|930|
+------+----+---+

+--------------+----+----+
|Senior Citizen|  No| Yes|
+--------------+----+----+
|           Yes| 666| 476|
|            No|4508|1393|
+--------------+----+----+

+-------+----+----+
|Partner|  No| Yes|
+-------+----+----+
|    Yes|2733| 669|
|     No|2441|1200|
+-------+----+----+

+----------+----+----+
|Dependents|  No| Yes|
+----------+----+----+
|       Yes|1521| 106|
|        No|3653|1763|
+----------+----+----+

+-------------+----+----+
|Phone Service|  No| Yes|
+-------------+----+----+
|          Yes|4662|1699|
|           No| 512| 170|
+-------------+----+----+

+----------------+----+---+
|  Multiple Lines|  No|Yes|
+----------------+----+---+
|             Yes|2121|850|
|No phone service| 512|170|
|              No|2541|849|
+----------------+----+---+

+----------------+----+----+
|Internet Service|  No| Yes|
+----------------+----+----+
|     Fiber optic|1799|1297

In [0]:
import pandas as pd

# 1. Convert association results to a Pandas DataFrame
assoc_df = pd.DataFrame(Feature_Cols)

# 2. Filter for statistically significant associations (p-value < 0.05)
significant_cols = assoc_df[assoc_df['p_value'] < 0.05].sort_values(by='cramers_v', ascending=False)

# Display the associated columns sorted by association strength (Cramer's V)
display(significant_cols)

column_1,column_2,chi_square,p_value,degrees_of_freedom,cramers_v,sample_size
Contract,Churn Label,1184.5965720837926,5.863038300673393E-258,2,0.4101156965761409,7043
Online Security,Churn Label,849.9989679615963,2.6611496351767036E-185,2,0.3474004326740552,7043
Tech Support,Churn Label,828.1970684587393,1.4430840279999813E-180,2,0.3429161982469257,7043
Internet Service,Churn Label,732.309589667794,9.571788222840544E-160,2,0.32245455521230887,7043
Payment Method,Churn Label,648.1423274814,3.6823546520098007E-140,3,0.30335862555407056,7043
Online Backup,Churn Label,601.8127901134089,2.079759216086546E-131,2,0.2923155121954445,7043
Device Protection,Churn Label,558.419369407389,5.505219496457244E-122,2,0.281579732968073,7043
Dependents,Churn Label,433.7343787644573,2.500972399855357E-96,1,0.24816074207383979,7043
Streaming Movies,Churn Label,375.66147934526555,2.667756755723757E-82,2,0.230950809069268,7043
Streaming TV,Churn Label,374.20394331098134,5.528994485739025E-82,2,0.23050233844668092,7043


In [0]:
significant_cols["column_1"].tolist()

['Contract',
 'Online Security',
 'Tech Support',
 'Internet Service',
 'Payment Method',
 'Online Backup',
 'Device Protection',
 'Dependents',
 'Streaming Movies',
 'Streaming TV',
 'Paperless Billing',
 'Senior Citizen',
 'Partner',
 'Multiple Lines']